# Revision-Aware Repair Verification for Gemma 4 Software Repair Agents

**Public reviewer companion · Clifford Imaguezegie · v1.5**

This notebook provides the project source snapshots and runs two CPU checks without model calls. The first runs the actual frozen controller and trusted pytest adapter on a synthetic repository. The second reconstructs the reported study outcomes from selected JSON records. Use a Python 3.12 CPU session with Git. The notebook includes the pinned offline wheels.

[Project evidence index](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/docs/evidence-index.md#e01) · [References](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/docs/references.md) · [Reproduction instructions](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/evidence/REPRODUCE.md)

## Study coverage

| Evidence group | Pairs | Runs | Outcome |
|---|---:|---:|---|
| GP03 primary autonomous | 9 | 18 | B and C each passed 6/9. Verified delivery was 5/9 for B and 4/9 for C. |
| GP04 and GP05 autonomous follow-up | 3 | 6 | B and C each passed 1/3. |
| GP04 controlled activation | 2 | 4 | C detected the two injected failures in 1.150 s and 1.047 s. |

Across all 24 autonomous runs, no trusted pass was followed by a qualifying edit. The primary paired difference was 0.0 percentage points with a 95% repository-stratified bootstrap interval of −22.2 to +22.2 points. The total of 7/12 protected passes per policy is descriptive across cohorts.

## CPU reproduction

Run the next two cells. The first contains the pinned archive payloads. The second verifies their SHA-256 values, installs pytest into a temporary folder using only the bundled wheels, and runs the replay and reconciliation. Repeating the cells creates a fresh output directory. Download the source archive from Kaggle's Output panel to inspect the study notebook copies offline.

- E17 package SHA-256 `3b8c15a8433909c09fab11cce28f136c8ef9d195e2769f237c24b71315cd96c0`
- Source notebook archive SHA-256 `f4964873ebeabd01ae5aec55760cb51772fbe12571a40d165cfdb8fc210bf52f`

In [1]:
E17_NAME = 'Gemma4_RARV_CPU_Review_Package_v1_0.zip'
PAYLOADS = {
    'Gemma4_RARV_CPU_Review_Package_v1_0.zip': ('...
"""),
    'Gemma4_RARV_Study_Notebook_Sources_v1_0.zip': ('...
"""),
}


In [2]:
from pathlib import Path
import base64
import hashlib
import json
import os
import subprocess
import sys
import tempfile
import venv
from zipfile import ZipFile

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Use a Python 3.12 CPU notebook.")
working = Path("/kaggle/working")
if not working.is_dir():
    working = Path.cwd()
run_dir = Path(tempfile.mkdtemp(prefix="rarv_cpu_review_", dir=working))

for name, (expected, encoded) in PAYLOADS.items():
    data = base64.b64decode(encoded)
    if hashlib.sha256(data).hexdigest() != expected:
        raise ValueError(f"Digest mismatch for {name}")
    (run_dir / name).write_bytes(data)

package_zip = run_dir / E17_NAME
with ZipFile(package_zip) as archive:
    archive.extractall(run_dir)
package = run_dir / package_zip.stem
venv_dir = run_dir / "pinned_env"
venv.EnvBuilder(with_pip=False).create(venv_dir)
review_python = venv_dir / "bin" / "python"
deps = subprocess.check_output(
    [str(review_python), "-I", "-c", "import sysconfig; print(sysconfig.get_path('purelib'))"],
    text=True,
).strip()
install = [
    sys.executable, "-m", "pip", "--isolated", "install", "--no-index",
    "--find-links", str(package / "wheels"), "--require-hashes",
    "--ignore-installed", "--no-compile", "--target", str(deps),
    "-r", str(package / "requirements.txt"),
]
installed = subprocess.run(install, capture_output=True, text=True, timeout=120)
if installed.returncode:
    raise RuntimeError(installed.stderr[-3000:])
env = {key: value for key, value in os.environ.items() if key not in {"PYTHONPATH", "PYTEST_ADDOPTS"}}
env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
commands = (
    ["replay_controller.py", "--output", str(run_dir / "replay")],
    ["reconcile_results.py", "--output", str(run_dir / "results.json")],
)
for command in commands:
    subprocess.run([str(review_python), *command], cwd=package, env=env, check=True, timeout=120)

report = json.loads((run_dir / "results.json").read_text())
assert report["autonomous_runs"] == 24 and report["controlled_runs"] == 4
assert report["qualifying_autonomous_pass_to_edit_events"] == 0
print("Both CPU checks passed. The source notebook archive and validation outputs are in", run_dir)


[fixture B] after edit STALE | broken gate blocked | restored gate accepted
[fixture C] after edit CURRENT_FAIL | broken gate blocked | restored gate accepted
PASS | actual controller and trusted pytest | B STALE, C CURRENT_FAIL | both gates block and restore
Trusted pytest checks 6 | model calls 0 | Report <local review directory>/replay/Replay_Report.json
{
  "GP03_primary_inference": {
    "assigned_pair_bounds": [
      0.0,
      0.0
    ],
    "bootstrap_draws": 10000,
    "bootstrap_seed": 20261108,
    "paired_counts": {
      "b_only": 1,
      "both_pass": 5,
      "c_only": 1,
      "neither_pass": 2
    },
    "paired_repo_stratified_bootstrap_95": [
      -0.2222222222222222,
      0.2222222222222222
    ],
    "protected_success_difference_c_minus_b": 0.0
  },
  "GP05_protected_counts": {
    "C": {
      "errors": 0,
      "failures": 1,
      "passed": 97,
      "skipped": 0,
      "tests": 98
    }
  },
  "autonomous_proactive_rechecks": 0,
  "autonomous_runs": 24,
  "

## Study notebooks

The linked notebooks provide source and methods for inspection. Their raw session outputs remain in the sealed study archives. The same released study notebook copies are included in the payload above.

| Notebook | Purpose |
|---|---|
| [GP00 Task Audit](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP00_Task_Audit.ipynb) | Audit the task inventory and construct the provisional development and evaluation split. |
| [GP01A Harness Check](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP01A_Harness_Check.ipynb) | Check that the organizer evaluator can distinguish a baseline failure from a reference pass. |
| [GP01B Reference Qualification](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP01B_Reference_Qualification.ipynb) | Qualify the development cases against independent baseline and reference outcomes. |
| [GP01C Qualification Retry](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP01C_Qualification_Retry.ipynb) | Recheck the development cases with outstanding infrastructure issues. |
| [GP02B Six Development Cases](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP02B_Six_Development_Cases.ipynb) | Inspect the six-case development batch and repair traces instrumentation. |
| [GP02C Revision Ledger](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP02C_Revision_Ledger.ipynb) | Exercise revision changes and verification receipt states on CPU. |
| [GP02C Trusted Pytest Adapter](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP02C_Trusted_Pytest_Adapter.ipynb) | Exercise the trusted pytest adapter and its provenance checks. |
| [GP02C Live Attachment](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP02C_Live_Attachment.ipynb) | Inspect the early organizer-worker attachment for the revision controller. |
| [GP03 CPU Qualification](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP03_CPU_Qualification.ipynb) | Qualify public test execution and recover environment-specific holds. |
| [GP03 Public Adapter](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP03_Public_Adapter.ipynb) | Verify the public test adapter against the selected nine tasks. |
| [GP03 Paired CPU Acceptance](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP03_Paired_CPU_Acceptance.ipynb) | Exercise the paired B and C paths and the synthetic fixture used by E17. |
| [GP03 Official Worker Acceptance](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP03_Official_Worker_Acceptance.ipynb) | Check the organizer worker boundary, inputs and attachment contract. |
| [GP03 Frozen Evaluation](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP03_Frozen_Evaluation.ipynb) | Inspect the frozen paired evaluation and accepted transport settings. |
| [GP03 Independent Grading](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP03_Independent_Grading.ipynb) | Independently grade archived production candidates in the organizer evaluator. |
| [GP03 Final Grade Recovery](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP03_Final_Grade_Recovery.ipynb) | Restore the missing offline dependencies and reconcile all nine primary pairs. |
| [GP04 Followup Evidence](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP04_Followup_Evidence.ipynb) | Read the GP04 controlled and additional autonomous results. |
| [GP05 Supplemental Pair](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/notebooks/GP05_Supplemental_Pair.ipynb) | Qualify, execute and independently grade the twelfth autonomous pair. |

## What the checks verify

The CPU replay verifies receipt and gate transitions in the actual controller. The reconciliation rebuilds outcomes from recorded protected verdicts. A fresh protected grade or Gemma run uses the organiser evaluator and licensed competition inputs. Authorised readers with the sealed source archives can follow the [full archive instructions](https://github.com/clifford-imaguezegie/gemma4-repair-verification/blob/main/evidence/REPRODUCE.md).